
# Upload a Folder to Amazon S3

This notebook function recursively uploads all files in a local folder to an **Amazon S3** bucket, preserving the folder structure as S3 object keys.

---

## What it does
- Scans a local directory tree and finds every file (`rglob("*")`)
- Builds S3 object keys **relative to the folder root** (keeps subfolders intact)
- Guesses and sets the **Content-Type** (e.g., `text/plain`, `application/json`) for better handling in S3
- Supports a **`prefix`** to place uploads under an S3 path like `datasets/text_chunks/...`
- Provides a **`dry_run=True`** mode to preview uploads without changing anything

---

## Prerequisites
- AWS credentials configured via one of:
  - Environment variables: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`
  - AWS config files: `~/.aws/credentials`, `~/.aws/config`
  - IAM role (if running on EC2/ECS/EKS/etc.)
- Python packages: `boto3`, `botocore`, `pathlib`, `mimetypes`


In [ ]:
import os
import sys
import boto3
from pathlib import Path
import mimetypes
from boto3.s3.transfer import TransferConfig
from botocore.exceptions import BotoCoreError, ClientError
from typing import Optional

# Add project root to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from config import REGION

# --------
# Config
# -------
FOLDER_PATH = "../data/text_chunks"
AWS_REGION = REGION
#BUCKET_NAME = "moj-analytical-platform-user-guidance"
BUCKET_NAME = "moj-de-user-guidance-kb-dev"

In [ ]:
# Initialize S3 client
s3 = boto3.client("s3", region_name=AWS_REGION)

In [ ]:
def upload_folder_to_s3(folder_path, bucket_name):
    folder = Path(folder_path)
  
    # Iterate recursively over all files under the folder (skips directories)
    for file_path in folder.rglob("*"):
        if file_path.is_file():
            # Build S3 key without spaces
            key = str(file_path.relative_to(folder)) # S3 key
            # Guess content type for better handling in S3
            content_type, _ = mimetypes.guess_type(file_path)
            extra_args = {"ContentType": content_type} if content_type else {}

            # Upload file to S3
            s3.upload_file(
                Filename=str(file_path),
                Bucket=bucket_name,
                Key=key,  # Correct argument name
                ExtraArgs=extra_args
            )
            print(f"Uploaded{file_path} ->  s3://{bucket_name}/{key}")

In [ ]:
upload_folder_to_s3(
    FOLDER_PATH,
    BUCKET_NAME,
)

In [ ]:
# Use this only if the above upload don't work
# Option-2 to upload 
# Enhanced version with dry-run, prefix, and error handling
def upload_folder_to_s3(
    folder_path: str,
    bucket_name: str,
    *,
    prefix: Optional[str] = None,
    dry_run: bool = False
) -> None:
    """
    Recursively upload all files in a local folder to an S3 bucket.

    Args:
        folder_path (str): Local root folder to upload.
        bucket_name (str): Target S3 bucket name.
        prefix (Optional[str]): Optional key prefix to prepend to each object key in S3.
                                Example: prefix="datasets/text_chunks" -> objects stored under that path.
        dry_run (bool): If True, only prints what would be uploaded without performing the actual upload.

    Behavior:
        - Builds object keys relative to the given folder (preserves subfolder structure).
        - Guesses Content-Type using `mimetypes` for better handling in S3.
        - Leaves spaces in keys as-is to avoid unintentionally changing paths (S3 supports spaces).
          If you prefer normalizing spaces, modify `key.replace(" ", "_")`.
        - Prints a line per uploaded (or would-upload) file.

    Notes:
        - Ensure your AWS credentials and region are configured (env vars, ~/.aws/credentials, or IAM role).
        - Consider adding server-side encryption (SSE) or ACLs via `ExtraArgs` if needed for your use case.
    """
    folder = Path(folder_path)

    # Iterate recursively over all files under the folder (skips directories)
    for file_path in folder.rglob("*"):
        if not file_path.is_file():
            continue

        # Build S3 key: relative path from folder root (preserves structure)
        relative_key = str(file_path.relative_to(folder))
        # Optionally prepend prefix if provided
        key = f"{prefix.rstrip('/')}/{relative_key}" if prefix else relative_key

        # Guess content type to help S3/clients serve files correctly (e.g., text/plain, application/json)
        content_type, _ = mimetypes.guess_type(file_path)
        extra_args = {"ContentType": content_type} if content_type else {}

        # For security/compliance, you might add encryption:
        # extra_args["ServerSideEncryption"] = "AES256"  # or KMS: "aws:kms" with "SSEKMSKeyId"

        if dry_run:
            # Simulate the upload without making changes (safe preview)
            print(f"[DRY-RUN] Would upload: {file_path} -> s3://{bucket_name}/{key} {extra_args or ''}")
            continue

        # Perform the actual upload with minimal error handling
        try:
            s3.upload_file(
                Filename=str(file_path),  # local file path
                Bucket=bucket_name,       # S3 bucket
                Key=key,                  # object key path
                ExtraArgs=extra_args      # optional content type / encryption / ACLs
            )
            print(f"Uploaded: {file_path} -> s3://{bucket_name}/{key}")
        except (BotoCoreError, ClientError) as e:
            # Log the error and continue with the next file
            print(f"Failed to upload {file_path}: {e}")


In [ ]:

# Preview (safe)
#upload_folder_to_s3("/path/to/local/folder", "my-bucket", prefix="datasets/text_chunks", dry_run=True)

# Upload for real
upload_folder_to_s3(FOLDER_PATH, "moj-analytical-platform-user-guidance", prefix="datasets/text_chunks", dry_run=False)
